# NOTEBOOK 3 — ML CLASSIFICATION
**Project:** Trust & Transparency in AI-Powered Recruitment Platforms

## Models
- Model 1: TF-IDF + Logistic Regression (baseline)
- Model 2: DistilBERT Fine-Tuned ← **selected as final model (best performance)**

**Input:** `../data/features/combined_jobs_features.csv`  
**Requires:** `../collected_data/emscad_dataset.csv`  
**Output:** `../data/predictions/combined_jobs_final_predictions.csv`  
**Models saved:** `../models/lr/`, `../models/bert/`

In [108]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay, average_precision_score
)
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from tqdm import tqdm

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

ROOT             = Path('..').resolve()
FEATURES_PATH    = ROOT / 'data' / 'features'    / 'combined_jobs_features.csv'
EMSCAD_PATH      = ROOT / 'collected_data'        / 'emscad_dataset.csv'
PREDICTIONS_PATH = ROOT / 'data' / 'predictions' / 'combined_jobs_final_predictions.csv'
LR_MODEL_PATH    = ROOT / 'models' / 'lr'         / 'model_lr_pipeline.joblib'
BERT_MODEL_PATH  = ROOT / 'models' / 'bert'
PLOTS_PATH       = ROOT / 'outputs' / 'plots'

for p in [PREDICTIONS_PATH.parent, LR_MODEL_PATH.parent, BERT_MODEL_PATH.parent, PLOTS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

LR_THRESHOLD            = 0.60
BERT_THRESHOLD          = 0.55
MIN_RULE_SCORE_FOR_FLAG = 2   # lowered from 5 → allows high-BERT-confidence jobs through

print(f"LR threshold   : {LR_THRESHOLD}")
print(f"BERT threshold : {BERT_THRESHOLD}")
print(f"Min rule score : {MIN_RULE_SCORE_FOR_FLAG}")

Device: cuda
LR threshold   : 0.6
BERT threshold : 0.55
Min rule score : 2


In [109]:
def clean_text(text, max_chars=1500):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'#(EMAIL|PHONE|URL)_Keyed_\w+#', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:max_chars]

In [110]:
# LOAD & PREPARE EMSCAD
print("LOADING EMSCAD TRAINING DATASET")

emscad = pd.read_csv(EMSCAD_PATH)
print(f"EMSCAD raw shape : {emscad.shape}")
print(f"Raw 'fraudulent' value counts:\n{emscad['fraudulent'].value_counts()}")

raw_type = emscad['fraudulent'].dtype
if raw_type == object:
    label_map = {'t': 1, 'f': 0, 'true': 1, 'false': 0, '1': 1, '0': 0}
    emscad['label'] = emscad['fraudulent'].str.strip().str.lower().map(label_map)
else:
    emscad['label'] = emscad['fraudulent'].astype(int)

emscad = emscad[emscad['label'].notna()].copy()
emscad['label'] = emscad['label'].astype(int)

fraud_rate = emscad['label'].mean()
print(f"\nLabel distribution: {emscad['label'].value_counts().rename({0: 'Legitimate', 1: 'Fake'}).to_dict()}")
print(f"Fraud rate: {fraud_rate*100:.1f}%")

if fraud_rate > 0.30:
    print("\n⚠ WARNING: Fraud rate > 30% — labels may be inverted! Check label_map.")
else:
    print(f"\n✓ Fraud rate {fraud_rate*100:.1f}% is within expected range (~4-5%)")

text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
for col in text_cols:
    if col not in emscad.columns:
        emscad[col] = ''

emscad['combined_text'] = (
    emscad['title'].fillna('')           + ' ' +
    emscad['company_profile'].fillna('') + ' ' +
    emscad['description'].fillna('')     + ' ' +
    emscad['requirements'].fillna('')    + ' ' +
    emscad['benefits'].fillna('')
)
emscad['combined_text'] = emscad['combined_text'].apply(clean_text)
emscad = emscad[emscad['combined_text'].str.strip() != ''].reset_index(drop=True)
print(f"EMSCAD rows after cleaning: {len(emscad)}")

X_all = emscad['combined_text'].tolist()
y_all = emscad['label'].tolist()

X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f"Train: {len(X_tr)}  |  Val: {len(X_val)}")

LOADING EMSCAD TRAINING DATASET
EMSCAD raw shape : (17880, 18)
Raw 'fraudulent' value counts:
fraudulent
f    17014
t      866
Name: count, dtype: int64

Label distribution: {'Legitimate': 17014, 'Fake': 866}
Fraud rate: 4.8%

✓ Fraud rate 4.8% is within expected range (~4-5%)
EMSCAD rows after cleaning: 17880
Train: 14304  |  Val: 3576


In [111]:
# LOAD SCRAPED UK DATA
df = pd.read_csv(FEATURES_PATH)
print(f"Scraped jobs loaded: {len(df)}")

df['combined_text'] = (
    df['job_title'].fillna('')    + ' ' +
    df['company_name'].fillna('') + ' ' +
    df['job_description'].fillna('')
)
df['combined_text_clean'] = df['combined_text'].apply(clean_text)

Scraped jobs loaded: 2722


In [ ]:
# MODEL 1: TF-IDF + LOGISTIC REGRESSION
print("MODEL 1: TF-IDF + LOGISTIC REGRESSION")

lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=15000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        C=1.0, max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='lbfgs', n_jobs=-1
    ))
])

lr_pipeline.fit(X_tr, y_tr)

y_pred_lr = lr_pipeline.predict(X_val)
y_prob_lr = lr_pipeline.predict_proba(X_val)[:, 1]

print("\n--- LR Validation Performance ---")
print(classification_report(y_val, y_pred_lr, target_names=['Legitimate', 'Fake']))
lr_auc = roc_auc_score(y_val, y_prob_lr)
lr_ap  = average_precision_score(y_val, y_prob_lr)
print(f"ROC-AUC       : {lr_auc:.4f}")
print(f"Avg Precision : {lr_ap:.4f}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr_pipeline, X_all, y_all, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"5-Fold CV AUC : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

lr_pipeline.fit(X_all, y_all)
joblib.dump(lr_pipeline, LR_MODEL_PATH)
print(f"Saved LR_MODEL")

df['lr_fraud_probability'] = lr_pipeline.predict_proba(df['combined_text_clean'])[:, 1].round(4)
df['lr_fraud_prediction']  = (df['lr_fraud_probability'] >= LR_THRESHOLD).astype(int)

print(f"\nLR fake jobs : {df['lr_fraud_prediction'].sum()} "
      f"({df['lr_fraud_prediction'].mean()*100:.1f}%)")

tfidf_vocab   = lr_pipeline.named_steps['tfidf'].get_feature_names_out()
lr_coefs      = lr_pipeline.named_steps['clf'].coef_[0]
top_fake_idx  = np.argsort(lr_coefs)[-20:][::-1]
top_legit_idx = np.argsort(lr_coefs)[:20]

MODEL 1: TF-IDF + LOGISTIC REGRESSION

--- LR Validation Performance ---
              precision    recall  f1-score   support

  Legitimate       0.99      0.98      0.99      3403
        Fake       0.74      0.90      0.81       173

    accuracy                           0.98      3576
   macro avg       0.87      0.94      0.90      3576
weighted avg       0.98      0.98      0.98      3576

ROC-AUC       : 0.9879
Avg Precision : 0.9207


In [ ]:
# MODEL 2: DistilBERT FINE-TUNING
print("MODEL 2: DistilBERT FINE-TUNING (3 epochs, max_len=128)")

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class JobDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts; self.labels = labels
        self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

class InferDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze()}

MAX_LEN = 128; BATCH_SIZE = 32; EPOCHS = 3

train_ds = JobDataset(X_tr, y_tr, tokenizer, MAX_LEN)
val_ds   = JobDataset(X_val, y_val, tokenizer, MAX_LEN)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

bert_model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
cw      = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float).to(device))
optimizer   = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_dl) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

print(f"\nTraining DistilBERT - {EPOCHS} epochs ({'GPU' if device.type=='cuda' else 'CPU'})...")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    bert_model.train()
    train_loss = 0
    for batch in tqdm(train_dl, desc=f"Training Epoch {epoch+1}"):
        ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        out  = bert_model(input_ids=ids, attention_mask=mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        train_loss += loss.item()
    print(f"Avg Train Loss: {train_loss / len(train_dl):.4f}")

In [ ]:
# BERT VALIDATION
bert_model.eval()
bert_probs_val, bert_labels_val = [], []

with torch.no_grad():
    for batch in tqdm(val_dl, desc='Validating'):
        ids  = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
        out  = bert_model(input_ids=ids, attention_mask=mask)
        probs = torch.softmax(out.logits, dim=1)[:, 1]
        bert_probs_val.extend(probs.cpu().numpy())
        bert_labels_val.extend(batch['label'].numpy())

bert_preds_val = [1 if p >= 0.5 else 0 for p in bert_probs_val]
bert_auc = roc_auc_score(bert_labels_val, bert_probs_val)
bert_ap  = average_precision_score(bert_labels_val, bert_probs_val)

print("\n--- DistilBERT Validation Performance (threshold=0.5) ---")
print(classification_report(bert_labels_val, bert_preds_val, target_names=['Legitimate', 'Fake']))
print(f"ROC-AUC       : {bert_auc:.4f}")
print(f"Avg Precision : {bert_ap:.4f}")

bert_model.save_pretrained(BERT_MODEL_PATH)
tokenizer.save_pretrained(BERT_MODEL_PATH)
print(f"Saved BERT_MODEL")

In [ ]:
# PREDICT ON SCRAPED DATA
infer_ds = InferDataset(df['combined_text_clean'].tolist(), tokenizer, MAX_LEN)
infer_dl = DataLoader(infer_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

bert_model.eval()
bert_probs_scraped = []

with torch.no_grad():
    for batch in tqdm(infer_dl, desc='Predicting scraped data'):
        ids  = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
        out  = bert_model(input_ids=ids, attention_mask=mask)
        probs = torch.softmax(out.logits, dim=1)[:, 1]
        bert_probs_scraped.extend(probs.cpu().numpy())

df['bert_fraud_probability'] = np.round(bert_probs_scraped, 4)

# Assign confidence label based on BERT probability
df['confidence_label'] = df['bert_fraud_probability'].apply(
    lambda p: 'High Confidence Fake' if p >= 0.7
    else ('Possibly Suspicious'       if p >= 0.4
    else 'Likely Legitimate')
)

# Final prediction: High Confidence Fake + Possibly Suspicious = fake
df['bert_fraud_prediction'] = (df['confidence_label'] != 'Likely Legitimate').astype(int)

print(f"
BERT fake jobs : {df['bert_fraud_prediction'].sum()} "
      f"({df['bert_fraud_prediction'].mean()*100:.1f}%)")
print(f"Confidence breakdown:
{df['confidence_label'].value_counts().to_string()}")

In [ ]:
# TRANSPARENCY OUTPUT
salary_95th = df['salary_avg'].quantile(0.95)

def transparency_output(row):
    if row['bert_fraud_prediction'] == 0:
        return 'No suspicious indicators detected.'
    reasons = str(row.get('flag_explanation', ''))
    if not reasons or reasons == 'No suspicious indicators detected.':
        reasons = 'Flagged by ML text pattern analysis'
    return (
        f"\u26a0 WARNING: This job was flagged as potentially suspicious.\n"
        f"Reasons     : {reasons}\n"
        f"BERT score  : {row['bert_fraud_probability']*100:.1f}%  |  "
        f"Risk Score  : {row['fraud_risk_pct']:.1f}%  |  "
        f"Risk Tier   : {row['risk_tier']}"
    )

df['transparency_output'] = df.apply(transparency_output, axis=1)

print("\n--- Top Flagged Jobs (BERT probability \u2265 0.4) ---")
flagged = df[df['bert_fraud_probability'] >= 0.4][
    ['job_title', 'company_name', 'source_platform',
     'bert_fraud_probability', 'confidence_label']
].sort_values('bert_fraud_probability', ascending=False).head(10)

if len(flagged):
    print(flagged.to_string(index=False))
else:
    print("No jobs above 0.4")

df.to_csv(PREDICTIONS_PATH, index=False)
print(f"\nSaved PREDICTIONS")

In [ ]:
# Plot 1: LR results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model 1: TF-IDF + Logistic Regression — Baseline (EMSCAD Validation)',
             fontsize=13, fontweight='bold')

ConfusionMatrixDisplay(
    confusion_matrix(y_val, y_pred_lr),
    display_labels=['Legitimate', 'Fake']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold')

fpr_lr, tpr_lr, _ = roc_curve(y_val, y_prob_lr)
axes[1].plot(fpr_lr, tpr_lr, color='steelblue', lw=2, label=f'AUC = {lr_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold'); axes[1].legend()

axes[2].barh(range(15), lr_coefs[top_fake_idx[:15]][::-1], color='tomato')
axes[2].set_yticks(range(15))
axes[2].set_yticklabels(tfidf_vocab[top_fake_idx[:15]][::-1], fontsize=8)
axes[2].set_title('Top 15 Words \u2192 Fake', fontweight='bold')
axes[2].set_xlabel('Coefficient')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '3_lr_results.png', bbox_inches='tight')
plt.close()
print("Saved \u2192 3_lr_results.png")

In [ ]:
# Plot 2: DistilBERT results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model 2: DistilBERT Fine-Tuned — Final Selected Model (EMSCAD Validation)',
             fontsize=13, fontweight='bold')

ConfusionMatrixDisplay(
    confusion_matrix(bert_labels_val, bert_preds_val),
    display_labels=['Legitimate', 'Fake']
).plot(ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title('Confusion Matrix', fontweight='bold')

fpr_b, tpr_b, _ = roc_curve(bert_labels_val, bert_probs_val)
axes[1].plot(fpr_b, tpr_b, color='mediumseagreen', lw=2, label=f'AUC = {bert_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold'); axes[1].legend()

axes[2].hist(df[df['bert_fraud_prediction']==0]['bert_fraud_probability'],
             bins=30, alpha=0.7, color='steelblue', label='Legitimate')
axes[2].hist(df[df['bert_fraud_prediction']==1]['bert_fraud_probability'],
             bins=30, alpha=0.7, color='tomato', label='Fake')
axes[2].axvline(BERT_THRESHOLD, color='black', linestyle='--', label=f'Threshold ({BERT_THRESHOLD})')
axes[2].set_xlabel('Fraud Probability'); axes[2].set_ylabel('Count')
axes[2].set_title('Probability Distribution\n(Scraped Data)', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.savefig(PLOTS_PATH / '3_bert_results.png', bbox_inches='tight')
plt.close()
print("Saved \u2192 3_bert_results.png")

In [ ]:
# Plot 3: Model comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Model Comparison: LR (Baseline) vs DistilBERT (Final)',
             fontsize=13, fontweight='bold')

models = ['LR (Baseline)', 'DistilBERT (Final)']
aucs   = [lr_auc, bert_auc]
bars   = axes[0].bar(models, aucs, color=['steelblue', 'mediumseagreen'], width=0.4)
axes[0].set_ylim(0.5, 1.05)
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('ROC-AUC Comparison\n(EMSCAD Validation)', fontweight='bold')
for b, v in zip(bars, aucs):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                 f'{v:.4f}', ha='center', fontweight='bold')

axes[1].plot(fpr_lr, tpr_lr, color='steelblue',      lw=2, label=f'LR  AUC={lr_auc:.3f}')
axes[1].plot(fpr_b,  tpr_b,  color='mediumseagreen', lw=2, label=f'DistilBERT AUC={bert_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve Comparison', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_PATH / '3_model_comparison.png', bbox_inches='tight')
plt.close()
print("Saved \u2192 3_model_comparison.png")

In [ ]:
# FINAL SUMMARY
print("=" * 55)
print("STEP 3 SUMMARY")
print("=" * 55)
print(f"Training dataset      : EMSCAD ({len(emscad):,} rows, {sum(y_all)} fraudulent)")
print(f"Scraped jobs analysed : {len(df):,}")
print(f"")
print(f"--- Model Performance (EMSCAD Validation) ---")
print(f"LR  \u2014 Val AUC         : {lr_auc:.4f}  |  AP: {lr_ap:.4f}")
print(f"BERT \u2014 Val AUC        : {bert_auc:.4f}  |  AP: {bert_ap:.4f}")
print(f"")
print(f"--- Selected Model: DistilBERT ---")
print(f"Inference threshold   : {BERT_THRESHOLD}")
print(f"Rule-guard minimum    : fraud_risk_score >= {MIN_RULE_SCORE_FOR_FLAG}")
hcf = df['confidence_label'].eq('High Confidence Fake').sum()
ps  = df['confidence_label'].eq('Possibly Suspicious').sum()
print(f"Fake jobs detected    : {df['bert_fraud_prediction'].sum()} ({df['bert_fraud_prediction'].mean()*100:.1f}%)")
print(f"  High Confidence Fake: {hcf}")
print(f"  Possibly Suspicious : {ps}")
print(f"  (Total: {hcf} + {ps} = {hcf+ps})")
print(f"LR reference count    : {df['lr_fraud_prediction'].sum()} ({df['lr_fraud_prediction'].mean()*100:.1f}%)")